# Clustering por atributos y lectura espacial — Siniestros fatales del Perú

**Pregunta:** ¿qué perfiles de siniestros fatales aparecen al agruparlos por gravedad y condiciones vehiculares, y dónde se distribuyen esos perfiles?

Se reproduce el **K-means original de cuatro grupos**. Después se agregan las variables excluidas y las coordenadas para interpretar los resultados, siguiendo la Parte II del laboratorio del miércoles y el uso de GeoPandas del lunes.

## 1. Método y selección de atributos

**K-means es clustering particional, no jerárquico.** Lo elegimos porque resume 9 106 siniestros en grupos interpretables de atributos numéricos preparados. Frente al **jerárquico aglomerativo**, es más sencillo de ejecutar a esta escala; frente a **DBSCAN espacial**, responde a la similitud entre siniestros, no a la concentración de ubicaciones. No presupone que los grupos sean geográficamente contiguos.

Entramos con **10 variables**: fallecidos, lesionados, número de vehículos, edad promedio; proporciones de vehículos con seguro, CITV y fugados; presencia de moto, conductor sin licencia y dosaje positivo.

**Excluimos del entrenamiento:**
- **Clase y causa:** no queremos reproducir categorías oficiales; además, muchas causas siguen en investigación.
- **Zona, peatón involucrado y coordenadas:** se reservan para contrastar los grupos y mapearlos; la ubicación no define la similitud.
- **Personas y conteos por sexo:** parcialmente redundantes con las víctimas y con algunas inconsistencias de registro.
- **Hora y trimestre:** requieren tratar su naturaleza temporal; **red vial:** sus códigos numéricos son categorías, no distancias.

Todas estas columnas permanecen en la matriz para **interpretarlas después**, sin cambiar las etiquetas que genera K-means.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import folium
from folium.plugins import MarkerCluster
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

# Archivo en la misma carpeta que este notebook.
df = pd.read_excel("matriz_analit.xlsx", index_col=0)
print("Matriz:", df.shape)
display(df[["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos", "edad_promedio"]].head(3))

## 2. Preparación — misma representación que el notebook original

Las proporciones comparan siniestros con distinto número de vehículos. Aplicamos `log1p` a fallecidos/lesionados, sustituimos dos edades registradas como cero por la mediana y estandarizamos las siete variables numéricas. Las tres binarias permanecen en 0/1.

In [ ]:
numericas = ["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos",
             "edad_promedio", "prop_seguro", "prop_citv", "prop_fugados"]
binarias = ["existe_moto", "existe_conductor_sin_licencia_vigente",
            "existe_dosaje_cualitativo_positivo"]

X = df[["cantidad_de_fallecidos", "cantidad_de_lesionados", "n_vehiculos",
        "edad_promedio"] + binarias].copy()
X["prop_seguro"] = df["n_con_seguro_vigente"] / df["n_vehiculos"]
X["prop_citv"] = df["n_con_citv_vigente"] / df["n_vehiculos"]
X["prop_fugados"] = df["n_vehiculos_fugados"] / df["n_vehiculos"]
X = X[numericas + binarias]

X.loc[X["edad_promedio"] == 0, "edad_promedio"] = X.loc[
    X["edad_promedio"] > 0, "edad_promedio"].median()
for columna in ["cantidad_de_fallecidos", "cantidad_de_lesionados"]:
    X[columna] = np.log1p(X[columna])
X[numericas] = StandardScaler().fit_transform(X[numericas])
print("Variables utilizadas:", X.shape[1], "| Registros:", len(X))

## 3. Evaluación de K y modelo

Comparamos **inercia** (menor es mejor), **silueta** (mayor es mejor) y **Davies–Bouldin** (menor es mejor). Mantenemos **K = 4**, como en el análisis original, para conservar sus perfiles; no afirmamos que sea el único valor óptimo.

In [ ]:
evaluacion = []
for k in range(2, 8):
    m = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    etiquetas = m.fit_predict(X)
    evaluacion.append([k, m.inertia_,
                       silhouette_score(X, etiquetas, sample_size=1500, random_state=42),
                       davies_bouldin_score(X, etiquetas)])

evaluacion = pd.DataFrame(evaluacion, columns=["K", "Inercia", "Silueta", "Davies-Bouldin"])
display(evaluacion.round(3))

In [ ]:
ax = sns.lineplot(data=evaluacion, x="K", y="Inercia", marker="o")
ax.set(title="Inercia según K", xlabel="K", ylabel="Inercia")
plt.show()


ax = sns.lineplot(data=evaluacion, x="K", y="Silueta", marker="o")
ax.set(title="Silhouette según K", xlabel="K", ylabel="Silhouette")
plt.show()


ax = sns.lineplot(data=evaluacion, x="K", y="Davies-Bouldin", marker="o")
ax.set(title="Davies-Bouldin según K", xlabel="K", ylabel="Davies-Bouldin")
plt.show()

In [ ]:
modelo = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)
df["cluster_kmeans"] = modelo.fit_predict(X)
print("Inercia con K=4:", round(modelo.inertia_, 2))
display(df["cluster_kmeans"].value_counts().sort_index().rename("Siniestros").to_frame())

## 4. Interpretación de los cuatro grupos

Interpretamos con las **variables originales**, no con sus valores estandarizados. Las descripciones se asignan después del algoritmo; los números 0–3 son etiquetas arbitrarias.

In [ ]:
perfil_datos = df.assign(
    prop_seguro=df["n_con_seguro_vigente"] / df["n_vehiculos"],
    prop_citv=df["n_con_citv_vigente"] / df["n_vehiculos"],
    prop_fugados=df["n_vehiculos_fugados"] / df["n_vehiculos"])
perfil = perfil_datos.groupby("cluster_kmeans").agg(
    siniestros=("cluster_kmeans", "size"),
    fallecidos_prom=("cantidad_de_fallecidos", "mean"),
    lesionados_prom=("cantidad_de_lesionados", "mean"),
    vehiculos_prom=("n_vehiculos", "mean"),
    edad_prom=("edad_promedio", "mean"),
    seguro_pct=("prop_seguro", "mean"),
    citv_pct=("prop_citv", "mean"),
    fugados_pct=("prop_fugados", "mean"),
    moto_pct=("existe_moto", "mean"),
    sin_licencia_pct=("existe_conductor_sin_licencia_vigente", "mean"),
    dosaje_pct=("existe_dosaje_cualitativo_positivo", "mean"))
perfil.loc[:, perfil.columns.str.endswith("_pct")] *= 100
display(perfil.round(1))

**Perfiles obtenidos:**

- **Clúster 0 — Mayor gravedad humana (1 165):** promedia **2,22 fallecidos** y **4,79 lesionados** por siniestro; predominan eventos con múltiples víctimas.
- **Clúster 1 — Fuga vehicular (885):** cerca del **98,8 %** de los vehículos involucrados figuran como fugados, en promedio.
- **Clúster 2 — Mayor cobertura vehicular (3 510):** registra **97,1 %** de vehículos asegurados y **75,7 %** con CITV vigente, en promedio.
- **Clúster 3 — Motocicletas y falta de licencia (3 546):** **59,1 %** de siniestros con motocicletas y **53,9 %** con al menos un conductor sin licencia vigente.

Estas características distinguen **perfiles dentro de siniestros fatales**; no demuestran causalidad ni probabilidad de sufrir un accidente.

## 5. Agregamos las variables excluidas para contrastar los perfiles

El algoritmo **no vuelve a entrenarse**: las clases, causas, zona, peatones y demografía solo describen qué registros quedaron dentro de cada grupo.

In [ ]:
excluidas = ["zona_RURAL", "peaton_involucrado", "clase_ATROPELLO",
             "clase_CHOQUE", "clase_DESPISTE", "causa_INVESTIGACION"]
contexto = df.groupby("cluster_kmeans")[excluidas].mean().mul(100)
contexto["personas_prom"] = df.groupby("cluster_kmeans")["n_personas"].mean()
contexto["hombres_prom"] = df.groupby("cluster_kmeans")["n_masculino"].mean()
contexto["mujeres_prom"] = df.groupby("cluster_kmeans")["n_femenino"].mean()
display(contexto.round(1))

# Distribución porcentual de las categorías que tampoco entraron a K-means.
for variable in ["hora_siniestro", "trimestre", "red_vial"]:
    print("Distribución (%) de", variable)
    display((pd.crosstab(df["cluster_kmeans"], df[variable], normalize="index") * 100).round(1))

**Qué aportan las variables excluidas:**

- **G0:** **82,5 %** en zona rural; al incorporar `clase_*`, aparecen principalmente **choques (50,6 %)** y **despistes (44,5 %)**. La mayor gravedad ya era parte de los atributos de entrenamiento; las clases y la zona son un contraste externo.
- **G1:** **92,2 %** son atropellos y **92,3 %** registran peatones involucrados; **76,6 %** conservan la causa *en investigación*. Se descubre esa asociación sin haber utilizado la clase ni el peatón para formar el grupo.
- **G2:** mezcla **choques (36,2 %)**, **atropellos (35,5 %)** y **despistes (23,1 %)**. Es un perfil de cobertura vehicular, **no** una clase específica de accidente.
- **G3:** prevalecen **choques (38,7 %)** y **despistes (34,5 %)**. Los atributos de moto y licencia ya influyeron en la agrupación; estas clases sirven para contextualizarla.

Las tablas por **hora, trimestre y red vial** muestran cómo se distribuyen sus códigos; no atribuimos significados horarios o viales sin disponer del diccionario de categorías. Los promedios por sexo requieren cautela por inconsistencias de registro.

## 6. Lectura espacial con GeoPandas

Igual que en el laboratorio: `X = longitud`, `Y = latitud` y `EPSG:4326`. **Las coordenadas se agregan después de K-means** para representar los mismos cuatro perfiles, no para construirlos. El archivo `peru_contorno.geojson` contiene un contorno nacional simplificado.

In [ ]:
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["coordenadas_longitud"],
                                    df["coordenadas_latitud"]), crs="EPSG:4326")
peru = gpd.read_file("peru_shapes/per_admbnda_adm1_ign_20200714.shp").to_crs(gdf.crs)
nombres = {0: "G0 · Mayor gravedad", 1: "G1 · Fuga vehicular",
           2: "G2 · Seguro y CITV", 3: "G3 · Motos y licencia"}
colores = {0: "#c0392b", 1: "#e69f00", 2: "#2874a6", 3: "#239b56"}

fig, ax = plt.subplots(figsize=(9, 9))
peru.plot(ax=ax, color="#f5f5f5", edgecolor="#8c8c8c", linewidth=.7)
for k in range(4):
    pts = gdf[gdf["cluster_kmeans"] == k]
    pts.plot(ax=ax, color=colores[k], markersize=3, alpha=.55,
             label=f"{nombres[k]} ({len(pts):,})")
ax.legend(loc="lower left", fontsize=9, markerscale=3, frameon=True)
ax.set_title("Siniestros fatales: clústeres por atributos en el mapa")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("mapa_4_clusters_peru.png", dpi=160)
plt.show()

In [ ]:
# Un mapa por perfil evita que los puntos de un grupo oculten a otro.
fig, axes = plt.subplots(2, 2, figsize=(11, 13))
for k, ax in enumerate(axes.flat):
    peru.plot(ax=ax, color="#fafafa", edgecolor="#999999", linewidth=.6)
    gdf[gdf["cluster_kmeans"] == k].plot(
        ax=ax, color=colores[k], markersize=4, alpha=.55)
    ax.set_title(f"{nombres[k]} · n = {(df.cluster_kmeans == k).sum():,}")
    ax.set_axis_off()
plt.tight_layout()
plt.savefig("mapas_por_grupo.png", dpi=150)
plt.show()

### Comparación espacial simple

Para cuantificar una diferencia visible en el mapa, dividimos el territorio en **franjas aproximadas de latitud**. **No son departamentos ni regiones administrativas** y no se usaron para entrenar.

In [ ]:
franjas = pd.cut(df["coordenadas_latitud"], bins=[-19, -14, -8, -3],
                 labels=["Sur (< -14°)", "Centro (-14° a -8°)", "Norte (> -8°)"])
distribucion = pd.crosstab(df["cluster_kmeans"], franjas, normalize="index") * 100
display(distribucion.round(1))

**Interpretación espacial:** el grupo **G3** tiene una presencia relativa mayor en la franja norte (**33,9 %**) que **G2 (16,8 %)**. En los cuatro grupos predomina la franja central según estos límites. Los mapas permiten observar dónde aparecen los perfiles, pero **no identifican zonas de mayor riesgo**: faltan medidas de exposición, como tránsito o población, y una delimitación administrativa más detallada.

La ubicación no modificó ningún grupo. Para estudiar *concentraciones físicas* habría que plantear por separado DBSCAN espacial o densidades geográficas.

## Visualización con Folium

### Mapa con MarketCluster

In [ ]:
gdf

In [ ]:
centro = [-9.2, -75.0]
tile = "Esri.WorldGrayCanvas"

map1 = folium.Map(
    location=centro,
    zoom_start=5,
    tiles=tile
)

grupo = MarkerCluster().add_to(map1)
for _, fila in gdf.iterrows():
    folium.Marker(
        location=[fila["coordenadas_latitud"], fila["coordenadas_longitud"]],
        popup=f"Grupo: {fila["cluster_kmeans"]}"
    ).add_to(grupo)

map1

### Mapa Coroplético

In [ ]:
regiones = peru.copy()
regiones["ADM1_ES"] = regiones["ADM1_ES"].replace({"Lima": "Lima y Callao", "Callao": "Lima y Callao"})
regiones = regiones.dissolve(by="ADM1_ES", as_index=False)

gdf_region = gpd.sjoin(
    gdf,
    regiones[["ADM1_ES", "ADM1_PCODE", "geometry"]],
    how="left",
    predicate="within"
)

conteo = gdf_region.groupby("ADM1_ES").size().rename("n_siniestros").reset_index()
regiones_conteo = regiones.merge(conteo, on="ADM1_ES", how="left")
regiones_conteo["n_siniestros"] = regiones_conteo["n_siniestros"].fillna(0).astype(int)

In [ ]:
mapa2 = regiones_conteo.explore(
    column="n_siniestros", 
    cmap="rocket_r",     
    tooltip=["ADM1_ES", "n_siniestros"],
    legend=True,
    tiles= tile,
    style_kwds={"color": "gray", "weight": 0.5}
)

mapa2

### Mapa coroplético de la región con más siniestros

In [ ]:
top_region = regiones_conteo.sort_values(by="n_siniestros", ascending=False).iloc[0, 0]

distritos = gpd.read_file("peru_shapes/per_admbnda_adm3_ign_20200714.shp").to_crs(gdf.crs)
distritos["ADM1_ES"] = distritos["ADM1_ES"].replace({"Lima": "Lima y Callao", "Callao": "Lima y Callao"})

distritos_top_region = distritos[distritos["ADM1_ES"]==top_region]

gdf_top_region = gpd.sjoin(
    gdf,
    distritos_top_region[["ADM3_ES", "ADM3_PCODE", "geometry"]],
    how="right",
    predicate="within"
)

conteo_distritos = gdf_top_region.groupby("ADM3_PCODE").size().rename("n_siniestros").reset_index()
distritos_conteo = distritos_top_region.merge(conteo_distritos, on="ADM3_PCODE", how="left")
distritos_conteo["n_siniestros"] = distritos_conteo["n_siniestros"].fillna(0).astype(int)

In [ ]:
mapa3 = distritos_conteo.explore(
    column="n_siniestros", 
    cmap="rocket_r",     
    tooltip=["ADM3_ES", "n_siniestros"],
    legend=True,
    tiles= tile,
    style_kwds={"color": "gray", "weight": 0.5}
)

mapa3

### DBSCAN con Folium

In [ ]:
crs_metros = gdf.estimate_utm_crs()
gdf_m = gdf.to_crs(crs_metros)
coords_m = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])

In [ ]:
gdf_m["cluster"] = DBSCAN(eps=30_000, min_samples=50).fit_predict(coords_m)

vecinos = NearestNeighbors(
    n_neighbors=50
)

vecinos.fit(coords_m)

distancias, _ = vecinos.kneighbors(coords_m)

k_distancias = distancias[:, -1]

k_distancias = np.sort(k_distancias)
kdist_df = pd.DataFrame({
    "observacion_ordenada": np.arange(len(k_distancias)),
    "k_distancia": k_distancias
})

ax = sns.lineplot(
    data=kdist_df,
    x="observacion_ordenada",
    y="k_distancia"
)

ax.set(
    title=f"Gráfico de {50}-distancias",
    xlabel="Observaciones ordenadas",
    ylabel=f"Distancia al vecino {50}"
)

plt.show()

In [ ]:
for eps in [40000, 45000, 50000, 55000, 60000]:
    gdf_m["cluster"] = DBSCAN(eps=eps, min_samples=50).fit_predict(coords_m)

    # cuántos grupos hay, sin contar el ruido.
    print(f"eps = {eps}".center(100, "-"))
    n_grupos = gdf_m["cluster"].nunique() - (1 if -1 in gdf_m["cluster"].values else 0)
    print("Grupos:", n_grupos)
    print("Ruido:", (gdf_m["cluster"] == -1).sum())

In [ ]:
gdf_m["cluster"] = DBSCAN(eps=45000, min_samples=50).fit_predict(coords_m)

# Mapa interactivo: un color por grupo, ruido en gris
gdf_c = gdf_m.to_crs("EPSG:4326")

# convertimos el número de grupo en texto para tratarlo como categoría; el -1 pasa a llamarse "ruido".
gdf_c["grupo"] = gdf_c["cluster"].astype(str)
gdf_c.loc[gdf_c["cluster"] == -1, "grupo"] = "ruido"

# Primero dibujamos el ruido
mapa_db = gdf_c[gdf_c["grupo"] == "ruido"].explore(
    color="lightgray", marker_kwds={"radius": 2},

    tiles=tile, name="ruido",
    tooltip=["grupo", "cantidad_de_fallecidos", "cantidad_de_lesionados"]
)

# Luego se dibuja los grupos sobre el mismo mapa
gdf_c[gdf_c["grupo"] != "ruido"].explore(
    m=mapa_db,                          # dibujamos sobre el mismo mapa
    column="grupo", categorical=True, cmap="Set2",
    marker_kwds={"radius": 3},
    tooltip=["grupo", "cantidad_de_fallecidos", "cantidad_de_lesionados"],
    legend=False, name="grupos DBSCAN"
)
mapa_db

## 7. Exportar resultados

Guardamos las etiquetas y la geometría para reproducir los mapas o continuar el análisis en QGIS/GeoPandas.

In [ ]:
df.to_csv("siniestros_kmeans_4grupos.csv", index=False)
gdf[["cluster_kmeans", "cantidad_de_fallecidos", "geometry"]].to_file(
    "siniestros_kmeans_4grupos.geojson", driver="GeoJSON")
print("Guardados: CSV, GeoJSON y dos mapas PNG")